In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "notebook_setup.py").exists()),
    Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_setup import get_dataset_path

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import ScalarFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable

# =============================================================================
# =============================================================================
V_tail_upp = 2.73

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 20

# =============================================================================
# =============================================================================
mat_path = get_dataset_path("CETL_data_processed.mat")
data = sio.loadmat(mat_path)

processed_Qd_mat  = data["processed_Qd_mat"].astype(float)   # (201, 3511)
processed_Vd_mat  = data["processed_Vd_mat"].astype(float)   # (201, 3511)

n_rows, n_cells = processed_Qd_mat.shape
Qmax_all = processed_Qd_mat[-1, :].ravel()  # shape: (n_cells,)

# =============================================================================
# =============================================================================
V_low, V_high = 2.7, 3.0
mu = (V_low + V_high) / 2.0
sigma = (V_high - V_low) / 3.0

rng = np.random.default_rng(999)
V_init = np.empty(n_cells, dtype=float)

for i in range(n_cells):
    while True:
        v = rng.normal(mu, sigma)
        if v > V_tail_upp:
            V_init[i] = v
            break

# =============================================================================
# =============================================================================
Vseg_low = V_tail_upp
Vseg_upp = V_tail_upp + 0.30
n_sub = 6

seg_edges = np.linspace(Vseg_low, Vseg_upp, n_sub + 1)
count_no_upstream = np.sum(V_init <= Vseg_low + (Vseg_upp - Vseg_low)/n_sub)

usable_counts = np.zeros(n_sub, dtype=int)
for k in range(n_sub):
    upper = seg_edges[k + 1]
    usable_counts[k] = np.sum(V_init > upper)

all_counts = np.concatenate([[count_no_upstream], usable_counts])
labels = ["None"] + [f"{k+1}" for k in range(n_sub)]
x_pos = np.arange(len(all_counts))

# =============================================================================
# =============================================================================
fig, axs = plt.subplots(2, 2, figsize=(16, 10))

ax_full = axs[0, 0]
ax_hist = axs[0, 1]
ax_seg  = axs[1, 0]
ax_bar  = axs[1, 1]

def add_colorbar_space(ax, sm=None, visible=False):
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.3) 
    if visible and sm:
        cbar = fig.colorbar(sm, cax=cax)
        cbar.set_label('Maximum capacity (Ah)', labelpad=10)
        cbar.outline.set_visible(False)
    else:
        cax.axis('off')

norm = Normalize(vmin=np.nanmin(Qmax_all), vmax=np.nanmax(Qmax_all))

# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# macaron_colors = ['#FFB3BA', '#FFDFBA', '#FFFFBA', '#BAFFC9', '#BAE1FF', '#D0BBE4']
# cmap1 = LinearSegmentedColormap.from_list('macaron', macaron_colors, N=256)

nature_colors = [
    "#313695",
    "#74add1",
    "#e0f3f8",
    "#fee090",
    "#f46d43",
    "#a50026",
]
cmap1 = LinearSegmentedColormap.from_list("nature_marine", nature_colors, N=256)

for i in range(n_cells):
    ax_full.plot(processed_Vd_mat[:, i], processed_Qd_mat[:, i],
                 color=cmap1(norm(Qmax_all[i])), linewidth=1.0, alpha=0.9)

ax_full.set_xlabel('Voltage (V)', labelpad=10)
ax_full.set_ylabel('Remaining capacity (Ah)', labelpad=10)
ax_full.grid(False)

sm1 = ScalarMappable(norm=norm, cmap=cmap1)
sm1.set_array([])

# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
bins = 30
ax_hist.hist(V_init, bins=bins, density=True, alpha=1.0, color="#9ecae1",
             edgecolor="black", linewidth=0.8, label="Simulated initial voltage")

x_vals = np.linspace(V_low - 3*sigma, V_high + 3*sigma, 600)
pdf_ideal = (1.0 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_vals - mu) / sigma) ** 2)
ax_hist.fill_between(x_vals, pdf_ideal, color="#6b6b6b", alpha=0.20, linewidth=0)

pdf_vals = (1.0 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_vals - mu) / sigma) ** 2)
label_pdf = f"Fitted normal PDF\n($\\mu={mu:.2f}$ V, $\\sigma={sigma:.2f}$ V)"
ax_hist.plot(x_vals, pdf_vals, color="black", linewidth=2.0, label=label_pdf)

ax_hist.set_xlabel("Initial voltage (V)", labelpad=10)
ax_hist.set_ylabel("Probability density", labelpad=10)
ax_hist.legend(frameon=False, loc="upper left", fontsize=16) 
ax_hist.grid(False)

# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
macaron_deep_colors = ["#FF9999", "#FFB266", "#FFD966", "#66D9A3", "#66B2FF", "#C08AD9"]
cmap2 = LinearSegmentedColormap.from_list("nature_marine", nature_colors, N=256)
V_cutoff = 2.5

for i in range(n_cells):
    Vd, Qd, V0 = processed_Vd_mat[:, i], processed_Qd_mat[:, i], V_init[i]
    if V0 <= V_cutoff: continue
    
    mask = (Vd >= V_cutoff) & (Vd <= V0)
    if not np.any(mask): continue

    ax_seg.plot(Vd[mask], Qd[mask], color=cmap2(norm(Qmax_all[i])), linewidth=1.0, alpha=0.9)

ax_seg.set_xlabel("Voltage (V)", labelpad=10)
ax_seg.set_ylabel("Remaining capacity (Ah)", labelpad=10)
ax_seg.grid(False)

sm2 = ScalarMappable(norm=norm, cmap=cmap2)
sm2.set_array([])

# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# macaron_bar_colors = ["#FFB3BA", "#FFDFBA", "#FFFFBA", "#BAFFC9", "#BAE1FF",
#                       "#E0BBE4", "#FFDFD3", "#CDE7BE", "#F1C0E8", "#B5EAD7"]
macaron_bar_colors = [
    "#C97C7C",  # muted red
    "#D89C6A",  # muted orange
    "#D6C27A",  # muted yellow
    "#8FB18F",  # muted green
    "#7FA6C7",  # muted blue
    "#A58AC0",  # muted purple
    "#C7968E",  # dusty rose
    "#A7B98A",  # olive green
    "#C38DB7",  # mauve
    "#76B7A8"   # teal green
]
colors_for_bars = [macaron_bar_colors[i % len(macaron_bar_colors)] for i in range(len(all_counts))]

ax_bar.bar(x_pos, all_counts, color=colors_for_bars)
ax_bar.set_xlabel("Upstream fragment index", labelpad=10)
ax_bar.set_ylabel("Number of available cells", labelpad=10)

ax_hist.set_ylim(0, 7)

ax_bar.set_xticks(x_pos)
ax_bar.set_xticklabels(labels, rotation=0)
ax_bar.grid(False)

y_formatter = ScalarFormatter(useMathText=True)
y_formatter.set_powerlimits((-2, 2))
ax_bar.yaxis.set_major_formatter(y_formatter)

# =============================================================================
# =============================================================================
for ax in [ax_hist, ax_full, ax_bar, ax_seg]:
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(True)

add_colorbar_space(ax_full, sm=sm1, visible=True)
add_colorbar_space(ax_hist, visible=False)
add_colorbar_space(ax_seg,  sm=sm2, visible=True)
add_colorbar_space(ax_bar,  visible=False)

fig.tight_layout(pad=1.0, w_pad=2.5, h_pad=0.5)
fig.savefig(f"combined_2x2_tail_boundary_swapped_Vtail_{V_tail_upp}.png", dpi=300, bbox_inches="tight")

plt.show()